# 8.5 Hyperparameter Tuning

本 notebook 使用不依賴額外套件的小型 grid search，比較 hidden units、dropout rate 與 learning rate。重點是建立可重現的調參流程與實驗紀錄。


## 1. 載入套件


In [ ]:
import os
import random
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.datasets import make_classification
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print(tf.__version__)


## 2. 建立三類分類資料


In [ ]:
X, y = make_classification(
    n_samples=1800,
    n_features=20,
    n_informative=10,
    n_redundant=4,
    n_classes=3,
    n_clusters_per_class=1,
    class_sep=1.2,
    flip_y=0.04,
    random_state=SEED,
)
X = X.astype('float32')
y = y.astype('int64')

x_train_full, x_test, y_train_full, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=SEED
)
x_train, x_valid, y_train, y_valid = train_test_split(
    x_train_full, y_train_full, test_size=0.2, stratify=y_train_full, random_state=SEED
)

scaler = StandardScaler()
x_train = scaler.fit_transform(x_train).astype('float32')
x_valid = scaler.transform(x_valid).astype('float32')
x_test = scaler.transform(x_test).astype('float32')

num_classes = len(np.unique(y))
print(x_train.shape, x_valid.shape, x_test.shape)
print('num_classes:', num_classes)


## 3. 建立可調參模型函式


In [ ]:
def build_model(units=64, dropout_rate=0.2, learning_rate=1e-3):
    tf.keras.backend.clear_session()
    tf.random.set_seed(SEED)
    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(x_train.shape[1],)),
        tf.keras.layers.Dense(units, activation='relu'),
        tf.keras.layers.Dropout(dropout_rate),
        tf.keras.layers.Dense(max(units // 2, 8), activation='relu'),
        tf.keras.layers.Dense(num_classes, activation='softmax'),
    ])
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy'],
    )
    return model


## 4. 設定小型搜尋空間


In [ ]:
search_space = [
    {'units': 32, 'dropout_rate': 0.1, 'learning_rate': 1e-3},
    {'units': 64, 'dropout_rate': 0.2, 'learning_rate': 1e-3},
    {'units': 128, 'dropout_rate': 0.3, 'learning_rate': 5e-4},
    {'units': 64, 'dropout_rate': 0.4, 'learning_rate': 5e-4},
]
search_space


## 5. 執行 Grid Search

每一組設定都使用相同資料切分、epoch、batch size 與 EarlyStopping。


In [ ]:
results = []
best_model = None
best_history = None
best_val_accuracy = -np.inf

for i, params in enumerate(search_space, start=1):
    print(f'Run {i}/{len(search_space)}: {params}')
    start_time = time.time()
    model = build_model(**params)
    history = model.fit(
        x_train,
        y_train,
        validation_data=(x_valid, y_valid),
        epochs=35,
        batch_size=32,
        callbacks=[
            tf.keras.callbacks.EarlyStopping(
                monitor='val_loss',
                patience=6,
                restore_best_weights=True,
            )
        ],
        verbose=0,
    )
    train_time = time.time() - start_time
    best_epoch = int(np.argmin(history.history['val_loss']) + 1)
    val_loss, val_accuracy = model.evaluate(x_valid, y_valid, verbose=0)
    row = {
        **params,
        'epochs_ran': len(history.history['loss']),
        'best_epoch': best_epoch,
        'val_loss': val_loss,
        'val_accuracy': val_accuracy,
        'params_count': model.count_params(),
        'train_seconds': round(train_time, 2),
    }
    results.append(row)

    if val_accuracy > best_val_accuracy:
        best_val_accuracy = val_accuracy
        best_model = model
        best_history = history

results_df = pd.DataFrame(results).sort_values('val_accuracy', ascending=False)
results_df


## 6. 視覺化調參結果


In [ ]:
plt.figure(figsize=(7, 4))
labels = [
    f"u={row.units}, d={row.dropout_rate}, lr={row.learning_rate:g}"
    for row in results_df.itertuples()
]
plt.barh(labels, results_df['val_accuracy'])
plt.xlabel('Validation Accuracy')
plt.title('Hyperparameter Search Results')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()


## 7. 使用最佳設定做 Test Set 評估


In [ ]:
best_config = results_df.iloc[0].to_dict()
print(best_config)

test_loss, test_accuracy = best_model.evaluate(x_test, y_test, verbose=0)
print({'test_loss': test_loss, 'test_accuracy': test_accuracy})

y_pred = best_model.predict(x_test, verbose=0).argmax(axis=1)
print(classification_report(y_test, y_pred, digits=3))


In [ ]:
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.plot(best_history.history['loss'], label='train')
plt.plot(best_history.history['val_loss'], label='valid')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Best Model Loss')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(best_history.history['accuracy'], label='train')
plt.plot(best_history.history['val_accuracy'], label='valid')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.title('Best Model Accuracy')
plt.legend()

plt.tight_layout()
plt.show()


## 8. 套用自己的資料

調參時請先固定資料切分與評估指標，並把每次實驗的設定與結果記錄成表格。第一輪通常先調 learning rate 與模型容量，第二輪再調 dropout、L2 或 batch size。test set 不應用來反覆挑模型，只保留做最後確認。
